# PLTR fundamentals: revenue growth and operating profitability

Compare financial information filed before the February 1, 2024 purchase
with information filed afterward and before May 2, 2025.

Data comes from `../data/fundamentals.csv` and `../data/pltr_prices.csv`.
The earlier quarters, starting in 2021, provide the history needed for growth
and trailing twelve-month calculations.

The current return calculation starts from the $16.33 purchase-date close.

In [160]:
import numpy as np, pandas as pd, json, sys, yfinance as yf, plotly.express as px, scipy.stats as sc, requests
start, end = "2023-05-01", "2025-05-02"
purchase_date = "2024-02-01"
PLTR = pd.read_csv("../data/pltr_prices.csv", index_col=0, parse_dates=True).sort_index()
purchase_price = PLTR.loc[purchase_date]["Close"]

In [161]:
fundamentals = pd.read_csv("../data/fundamentals.csv", parse_dates=["period_end", "filed"]).sort_values("period_end")

fundamentals["operating_margin_ttm_pct"] = fundamentals["operating_income_usd"].rolling(4).sum() / fundamentals["revenue_usd"] * 100
fundamentals["revenue_growth_pcd"] = fundamentals["revenue_usd"].pct_change() * 100
fundamentals["operating_margin_pct"] = fundamentals["operating_income_usd"]  / fundamentals["revenue_usd"] * 100
fundamentals["revenue_ttm_usd"] = fundamentals["revenue_usd"].rolling(4).sum()

fundamentals = fundamentals[fundamentals["filed"] <= end]

known_at_purchase = fundamentals[fundamentals["filed"] < purchase_date]
analysis = fundamentals.loc[fundamentals["filed"].between(purchase_date, end, inclusive="left")].copy()

display(analysis.head())

,period_end,eps_diluted_usd,operating_income_usd,revenue_usd,filed,operating_margin_ttm_pct,revenue_growth_pcd,operating_margin_pct,revenue_ttm_usd
11,2023-12-31,NaN,65794000,608350000.0,2024-02-20,19.719898,8.992241,10.815156,2.225012e+09
12,2024-03-31,0.04,80881000,634338000.0,2024-05-07,31.013750,4.271883,12.750458,2.334164e+09
13,2024-06-30,0.06,105339000,678134000.0,2024-08-06,43.058894,6.904206,15.533656,2.478981e+09
14,2024-09-30,0.06,113140000,725516000.0,2024-11-05,50.330248,6.987115,15.594418,2.646338e+09
15,2024-12-31,NaN,11043000,827519000.0,2025-02-18,37.510075,14.059373,1.334471,2.865507e+09


## Revenue growth

The rates below are compound average **quarterly** revenue growth rates:
`(last revenue / first revenue) ** (1 / (number of quarters - 1)) - 1`.
The data contains consecutive quarters, so each row-to-row interval is one
quarter.

1. Was revenue growth accelerating or slowing?
2. Were operating margins improving?
3. How did these measures change from the latest quarter known at purchase to the latest available by end?

`Revenue growth`

In [162]:
total_growth_pre = (known_at_purchase.iloc[-1]["revenue_usd"] / known_at_purchase.iloc[0]["revenue_usd"])
m = len(known_at_purchase) - 1
avg_growth_pre = (total_growth_pre ** (1 / m) - 1) * 100

m = len(analysis) - 1
total_growth_post = (analysis.iloc[-1]["revenue_usd"] / analysis.iloc[0]["revenue_usd"])
avg_growth_post = (total_growth_post ** (1 / m) - 1) * 100

fig = px.line(known_at_purchase, x="filed", y="revenue_usd", labels={"index":"Date", "revenue_usd":"Revenue", "filed":"Date"}, title="Revenue growth").update_traces(name="Pre-Purchase", showlegend=True)

fig.add_annotation(
    x=0.3, y=0.45, xref="paper", yref="paper", showarrow=False,
    text=f"Pre: {avg_growth_pre:.2f}% avg growth per quarter"
).add_annotation(
    x=0.92, y=0.83, xref="paper", yref="paper", showarrow=False,
    text=f"Post: {avg_growth_post:.2f}% avg growth per quarter"
).add_vline(
    x=purchase_date, line_color="red", line_width=3,
    annotation_text=f"Purchase",
    annotation_position=f"top left",
    annotation_font_color="red")

post_plot = pd.concat([known_at_purchase.tail(1), analysis])

fig.add_scatter(
    x=post_plot["filed"], y=post_plot["revenue_usd"],
    mode="lines", name="Post-Purchase"
)

fig.show()

## Quarterly operating profitability

Operating margin is operating income divided by revenue. Its level is shown
in percent; changes in that level are measured in **percentage points (pp)**.
For example, moving from a 5% margin to a 7% margin is a 2 pp improvement.

The annotations show the mean quarter-to-quarter change within each group,
not a compound percentage growth rate. This remains meaningful when margins
are negative or cross zero.

`Margins`

In [163]:
avg_change_pre = known_at_purchase["operating_margin_pct"].diff().mean()
avg_change_post = analysis["operating_margin_pct"].diff().mean()

fig = px.line(known_at_purchase, x="filed", y="operating_margin_pct", labels={"index":"Date", "operating_margin_pct":"Operating margin (%)", "filed":"Date"}, title="Operating Margin change").update_traces(name="Pre-Purchase", showlegend=True)

fig.add_annotation(
    x=0.3, y=0.4, xref="paper", yref="paper", showarrow=False,
    text=f"Pre: {avg_change_pre:.2f}% pp per quarter"
).add_annotation(
    x=0.94, y=0.8, xref="paper", yref="paper", showarrow=False,
    text=f"Post: {avg_change_post:.2f}% pp per quarter"
).add_vline(
    x=purchase_date, line_color="red", line_width=3,
    annotation_text=f"Purchase",
    annotation_position=f"top left",
    annotation_font_color="red")

post_plot = pd.concat([known_at_purchase.tail(1), analysis])

fig.add_scatter(
    x=post_plot["filed"], y=post_plot["operating_margin_pct"],
    mode="lines", name="Post-Purchase"
)
fig.show()

### Understanding the Q4 2024 margin decline

Q4 2024 GAAP operating income was **$11.043 million** on revenue of
**$827.519 million**, giving an operating margin of approximately **1.33%**.
The sharp decline needs context before drawing conclusions about profitability.

Palantir disclosed **$115.776 million** in accelerated stock-based compensation
related to market-vesting stock appreciation rights (SARs), plus **$15.528
million** in related employer payroll taxes. Excluding those specific charges,
the company reported operating income of **$142.347 million** and a rounded
operating margin of **17%**. That is a company-reported non-GAAP comparison.
The GAAP series above retains the expenses.

Source: [Palantir Q4 2024 earnings release, including the reconciliation](https://www.sec.gov/Archives/edgar/data/1321655/000132165525000007/a2024q4ex991earningsrelease.htm).

## Trailing twelve-month (TTM) 
TTM revenue sums the latest four consecutive quarters. TTM operating margin
divides the operating income summed over those quarters by their total
revenue. These measures reduce seasonal differences and put growth
and profitability on a comparable annual basis.

The comparison below uses the latest SEC information available before purchase
and before the exclusive analysis cutoff.

Together with margin analysis, it shows whether the stock’s rise accompanied stronger business performance.

In [ ]:
before = known_at_purchase.iloc[-1]
after = fundamentals.iloc[-1]

summary = fundamentals.loc[
    [before.name, after.name],
    ["period_end", "filed", "revenue_ttm_usd", "operating_margin_ttm_pct"]
].copy()
summary.index = ["Known at purchase", "Latest before cutoff"]
display(summary)

revenue_growth = (after["revenue_ttm_usd"] / before["revenue_ttm_usd"] - 1) * 100
margin_change = after["operating_margin_ttm_pct"] - before["operating_margin_ttm_pct"]

PLTR = pd.read_csv(
    "../data/pltr_prices.csv", index_col=0, parse_dates=True
).sort_index()
prices = PLTR.loc[(PLTR.index >= purchase_date) & (PLTR.index < end), "Close"]
stock_return = (prices.iloc[-1] / purchase_price - 1) * 100

print(f"Stock-price endpoint: {prices.index[-1]:%Y-%m-%d}")
print(f"Stock return from ${purchase_price:.2f}: {stock_return:.1f}%")
print(f"TTM revenue growth: {revenue_growth:.1f}%")
print(f"TTM operating margin: {before['operating_margin_ttm_pct']:.2f}%"
      f" -> {after['operating_margin_ttm_pct']:.2f}%")
print(f"TTM margin change: {margin_change:+.2f} pp")

,period_end,filed,revenue_ttm_usd,operating_margin_ttm_pct
Known at purchase,2023-09-30,2023-11-03,2.125286e+09,6.511765
Latest before cutoff,2024-12-31,2025-02-18,2.865507e+09,37.510075


Stock-price endpoint: 2025-04-30
Stock return from $16.33: 625.3%
TTM revenue growth: 34.8%
TTM operating margin: 6.51% -> 37.51%
TTM margin change: +31.00 pp


## Conclusions

From the $16.33 purchase on 2024-02-01 through 2025-04-30, the stock returned 625.3%, while TTM revenue grew 34.8%.

TTM GAAP operating margin changed from 6.51% to 37.51%: a change of 31.00%. The Q4 2024 quarterly margin decline includes accelerated equity-compensation expenses and related payroll taxes.

The stock return exceeded TTM revenue growth. Comparing these measures motivates examining valuation and expectations for future growth.

This analysis describes business performance alongside the realized return. It does not establish a causal explanation of the stock-price movement.